In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from psilia.transforms import Transform
from psilia.vision import CameraIntrinsics, compute_uv_grid
from psilia import get_console
from psilia.nbx.gaussian_splatting import (
    sample_covariance,
    sample_from_scene,
    GaussianScene,
    project_gausians,
    bounding_boxes,
    quantize_and_clip_bbs,
    memory_layout,
    _sample_image,
    rasterize,
)

console = get_console()

key = jax.random.PRNGKey(0)

intr = CameraIntrinsics.load("../notebooks/_intr.yaml").resize(0.1)
W = intr.w
H = intr.h
console.print(intr)

In [ ]:
project = jax.jit(lambda x, cam,: screen_from_camera(cam.inv()(x), intr))
unproject = jax.jit(lambda x: _unproject(x, intr))
render = jax.jit(
    lambda xs, vs, down=1: _render(xs, vs, intr, (intr.h, intr.w), down, jnp.array(1.)),
    static_argnums=(2,))
compute_depths = jax.jit(lambda xs, cam: cam.inv()(xs)[:,2])

In [ ]:


# Camera pointing along the x-axis
cam = Transform.from_te(
    jnp.array([-10, 0., 0.]),
    [-90, 0, -90], seq="xyz", degrees=True
)
# Sample a scene of N Gaussians (means, covariances, colors)
N = 100
key, key0, key1, key2 = jax.random.split(key, 4)
mus = jax.random.uniform(key0, (N,3), minval=-5, maxval=5)
covs = sample_covariance(key1, N, smin=jnp.array([0.1, 0.1, 0.05]), smax=jnp.array([0.5, 2., 1.]))
cols = jax.random.uniform(key2, (N,3), minval=0., maxval=1.)
console.inspect(mus=mus, covs=covs)

# Camera
tf = Transform.id()

In [ ]:
import numpy as np
arr = np.load("exp/_point_cloud.npy", allow_pickle=True).item()

mus = arr["xs"]
N = mus.shape[0]
cs = arr["cs"][...,:3]/255
covs = 0.01*jnp.full((N,3,3), jnp.eye(3))
console.inspect(mus=mus, cs=cs)

sub = np.random.randint(0, N, 1_000)



In [ ]:
mus = mus[sub]
cs = cs[sub]
covs = covs[sub]
N = mus.shape[0]

plt.scatter(mus[:,0], mus[:,1], c=cs)


In [ ]:


# Gaussian Scene
scene = GaussianScene(mus, covs, cols, 1.*jnp.ones_like(mus[...,0]))
(mus2d, covs2d), valid = project_gausians(mus, covs, tf, intr)
scene2d = GaussianScene(mus2d, covs2d, cols, scene.weights)

console.inspect(mus2d=mus2d, covs2d=covs2d)
console.print(f"{valid.sum()} valid Gaussians")

# Memory Layout
bbs = bounding_boxes(mus2d, covs2d)
console.inspect(bbs=bbs)
bbs = quantize_and_clip_bbs(bbs, intr.w, intr.h)
console.inspect(bbs=bbs)
whs = bbs[:,1]-bbs[:,0]
areas = whs[:,0]*whs[:,1]
M = int(areas.sum())
ms, inds, off = memory_layout(areas, M)

console.inspect(areas=areas, M=M, ms=ms, inds=inds, off=off)

In [ ]:
# Rasterize
zs = compute_depths(mus, tf)
rasterize_result = rasterize(bbs, zs, M, W, H)
console.inspect(rasterize_result_counts=rasterize_result._counts)

In [ ]:



# UVs
uv_grid_flat = compute_uv_grid(intr, flat=True, indexing="uv")

# Pseudo Render image
sampling_context = (tf, uv_grid_flat, scene, zs, rasterize_result, intr)
im = _sample_image(key, *sampling_context)

# ==========================
fig, axs = plt.subplots(1,3, figsize=(8,4))
axs[0].imshow(im)
axs[1].imshow(rasterize_result._counts2d.transpose(1,0))
axs[2].hist(rasterize_result._counts, bins=100);
fig.show()

In [ ]:
key,_ = jax.random.split(key)
keys = jax.random.split(key, 1_000)


sampling_context = (tf, uv_grid_flat, scene, zs, rasterize_result, intr)
sample_many = jax.vmap(
    lambda key: _sample_image(key, *sampling_context))


im = _sample_image(key, *sampling_context)
ims = sample_many(keys)

# ==========================
fig, ax = plt.subplots(1,1, figsize=(8,4))
ax.imshow(ims.mean(0))


In [ ]:
import numpy as np
from psilia.gaussians import get_scales_and_quat
from psilia.plotting import RerunLogger


rrl = RerunLogger()



arr = np.load("exp/_point_cloud.npy", allow_pickle=True).item()

mus = arr["xs"]
cols = arr["cs"][...,:3]/255
N = mus.shape[0]
covs = (0.01**2)*jnp.full((N,3,3), jnp.eye(3))
console.inspect(mus=mus, cols=cols)

sub = np.random.randint(0, N, 1_000)
mus = jnp.array(mus[sub])
cols = jnp.array(cols[sub])
covs = jnp.array(covs[sub])
N = mus.shape[0]

scales, quats = get_scales_and_quat(covs)

In [ ]:

rrl.init("gaussian_splatting")


rrl.log_ellipsoid("scene", mus, scales, quats, colors=cols)